In [1]:
import os
import sys

from pymed_paperscraper import PubMed

sys.path.append(os.path.abspath(os.path.join("..")))
from operator import add
from typing import Annotated, TypedDict

from langchain_core.messages import SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from pydantic import BaseModel, Field

from utils.models import ModelName, Models


In [2]:
models = Models()
llm = models.create_nvidia_model(
    model_name=ModelName.NEMOTRON_3_5_SUPER_VL_120b_a12b.value
)

Creating model nvidia/nemotron-3.5-super-vl-120b-a12b


In [2]:
# Delete later
pubmed = PubMed(tool="langchain", email="jason@aol.com")
response = pubmed.query("perimenopause and stress", max_results=5)
results = [a.toDict() for a in response]


DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): eutils.ncbi.nlm.nih.gov:443
DEBUG:urllib3.connectionpool:https://eutils.ncbi.nlm.nih.gov:443 "GET /entrez/eutils/esearch.fcgi?tool=langchain&email=jason%40aol.com&db=pubmed&term=perimenopause+and+stress&retmax=5&retmode=json HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): eutils.ncbi.nlm.nih.gov:443
DEBUG:urllib3.connectionpool:https://eutils.ncbi.nlm.nih.gov:443 "GET /entrez/eutils/efetch.fcgi?tool=langchain&email=jason%40aol.com&db=pubmed&id=43015104&id=43013592&id=42979921&id=42976040&id=42973544&retmode=xml HTTP/1.1" 200 None


In [ ]:
# Delete later
titles = []
for i in range(len(results)):
    titles.append(
        f"<document>title:{results[i]['title']} \n\n abstract:{results[i]['abstract']}\n\nconclusions: {results[i]['conclusions'] or 'none'}</document>"
    )
len(titles)

5

In [8]:
print(titles[1])

<document>title:The Skin-Mood Axis in Midlife: Neuro-Immuno-Endocrine Cross-Talk in Perimenopausal and Menopausal Aesthetic Dermatology. 

 abstract:Perimenopause and menopause may alter skin quality, repair capacity, sleep, mood, self-perception, and aesthetic motivation, yet midlife aesthetic aging is often framed primarily around estrogen and collagen loss.
To propose a midlife skin-mood axis that integrates cutaneous biology, neuroendocrine signaling, immune activity, and psychosocial experience in perimenopausal and menopausal aesthetic dermatology.
This narrative review synthesizes evidence regarding menopausal hormonal change, barrier and extracellular matrix remodeling, stress and sleep physiology, oxytocin-related signaling, body image, observer perception, and aesthetic motivation.
Midlife skin concerns may reflect interactions among reduced barrier resilience, altered repair capacity, neuroendocrine stress pathways, and changes in self-perception rather than isolated tissue 

In [ ]:
documents = []
for i in range(len(results)):
    doc = f"title: {results[i]['title']} \n\nabstract: {results[i]['abstract']} \n\n conclusions:{results[i]['conclusions'] or 'none'}"
documents.append(doc)
print(doc)

title: Plant-derived nutrition and perimenopausal emotional health: current evidence and research priorities. 

abstract: Perimenopause combines ovarian hormone variability with changes in stress regulation, inflammatory-redox balance, neuroplasticity, and gut-brain metabolism, creating a period of increased vulnerability to depressive symptoms, anxiety, irritability, and sleep disturbance. This narrative review synthesizes evidence on plant-derived foods, bioactives, dietary supplements, and standardized botanical preparations relevant to emotional health during the menopausal transition. Targeted searches of PubMed/MEDLINE, Web of Science, and Scopus identified literature published through 31 July 2026. Evidence was classified by consumed form and reproductive-stage directness and integrated across three regulatory axes: endocrine-stress regulation, immune-redox-neuroplasticity, and gut-brain-metabolic communication. Human evidence includes small randomized trials of standardized saf

In [ ]:
def pubmed_search(query: str, max_results: int = 3):
    pubmed = PubMed(tool="langchain", email="jason@aol.com")
    response = pubmed.query(query=query, max_results=max_results)
    results = [a.toDict() for a in response]
    documents = []
    for i in results:
        doc = f"title: {results[i]['title']} \n\nabstract: {results[i]['abstract']} \n\n conclusions:{results[i]['conclusions'] or 'none'}"
        documents.append(doc)
    return documents

In [ ]:
search_results = pubmed_search("perimenopause and health issues in women over 40")

In [26]:
topic = "perimenopause and health issues in women over 40"

In [ ]:
class OverallState(TypedDict):
    topic: str
    pubmed_results: list[str]
    llm_reports: Annotated[list, add]
    final_report: str
    max_results: int


class PubMedReport(TypedDict):
    """The schema of the report llm expects and the pubmed_search output"""

    document: str
    topic: str


class AnalystReport(BaseModel):
    """Structure for  the Analyst  to Create/Generate a report"""

    title: str = Field("Title for analyst's report")
    analysis: str = Field("Analysis about the report")
    summary: str = Field("Summary of the report")


# For Writer to generate structured output
class WriterReport(BaseModel):
    """Structure for the final report"""

    title: str = Field(description="Title of the final report")
    report: str = Field(
        description="Final report based on all the reports generated by the analyst"
    )
    conclusion: str = Field(description="Final conclusion of the analysis")


In [ ]:
LLM_ANALYST_INSTRUCTIONS = """You are an expert analyst in the field of topic:{topic}. 
    You are tasked with reviewing a report's/research papers title,abstract,conclusions,results.
    The report is the format each <document> title abstract conclusions results </document>
    Instructions:
    1. Read the  document i.e.,title,abstract,conclusions and results carefully and precisely.In documents with no conclusions and/or results, consider only the title and abstract
    2. Analyze and review the  document and gather all the necessary and critical details in the data 
    3. Create a report based on all document 
    4. Do not bring outside inferences,assumptions or any information
    5.You report should be based on the document that is available to you and what you analyzed and concluded on the basis of your analysis
    6. The report should be between 100-200 words 
        - One clear Title
        - sub-headers wherever required 
    The report should based on only on what is included in the data you read.
    """


In [ ]:
def pubmed_search(state: OverallState):
    query = state["topic"]
    max_results = state["max_results"]
    pubmed = PubMed(tool="langchain", email="jason@aol.com")
    response = pubmed.query(query=query, max_results=max_results)
    results = [a.toDict() for a in response]
    documents = []
    for i in results:
        doc = f"title: {results[i]['title']} \n\nabstract: {results[i]['abstract']} \n\n conclusions:{results[i]['conclusions'] or 'none'}"
        documents.append(doc)
    return {"pubmed_results": documents}


In [ ]:
def call_llm(state: PubMedReport):
    topic = state["topic"]
    prompt = LLM_ANALYST_INSTRUCTIONS.format(topic=topic)
    response = llm.with_structured_output(AnalystReport).invoke(prompt)
    return {"llm_reports": [response.reports]}  # pyright:ignore


In [ ]:
def should_call_llm(state: OverallState):
    topic = state["topic"]
    return [
        Send("call_llm", {"topic": topic, "document": doc})
        for doc in state["llm_reports"]
    ]